In [70]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}

div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:40px;}
</style>
"""))

# 5. 생성형 AI 평가 : 
- 첫번째 체인 :  나라이름 -> 그 나라에서 가장 유명한 음식 
- 두번째 체인 :  음식 -> 음식의 레시피
- 최종 체인 : 나라이름 -> 그 나라에 가장 유명한 음식의 레시피

In [58]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser  #aimessage만 출력하는 parse
from langchain_core.output_parsers import JsonOutputParser
llm= ChatOllama(model="exaone3.5:2.4b")

#첫번째 체인 :  나라이름 -> 그 나라에서 가장 유명한 음식 
food_prompt_template = PromptTemplate(
    template="{국가}에서 가장 유명한 음식이 무엇입니까? Return the name of the food name ONLY.",
    input_variables =['국가'])
outputParser=StrOutputParser()

In [71]:
outputParser.invoke(llm.invoke(food_prompt_template.invoke({'한국'})))

'비빔밥'

In [60]:
famous_food_chain= food_prompt_template | llm | outputParser
famous_food_chain.invoke({'국가':'한국'})

'김치'

In [66]:
#두번째 체인 :  음식 -> 음식의 레시피
recipe_prompt_template = PromptTemplate(
                    template="""Give following information about {food_name},
                              1.Food name
                              2.Prep
                              3.Cook
                              4.Finish  
                Return ONLY a valid JSON object with no additional text.
                Example format:
                {{"food": "{food_name}"
                "1.Prep":"Thinly slice the vegetables.", 
                "2.Cook":"Stir-fry the vegetables and beef separately, then fry an egg.", 
                "3.Finish":"Top warm rice with toppings, fried egg, gochujang, and sesame oil."}}""",
                    input_variables =['food_name']
                    )

In [67]:
recipe_output_parser=JsonOutputParser()
recipe_output_parser.invoke(llm.invoke(recipe_prompt_template.invoke({'food_name':'김치'})))

{'food': '김치',
 '1.Prep': 'Chopped kimchi ingredients (daikon, napa cabbage, garlic, ginger) mixed with spices and salt.',
 '2.Cook': 'Fermented mixture left to mature, then chopped vegetables are simmered until partially cooked.',
 '3.Finish': 'Served on top of rice or noodles with optional garnishes like green onions and sesame seeds.'}

In [68]:
recipe_chain= recipe_prompt_template | llm | recipe_output_parser
recipe_chain.invoke({'food_name':'김치'})

{'food': '김치',
 '1.Prep': 'Soak garlic, ginger, and scallions in water, then crush them using a mortar and pestle. Mix kimchi ingredients including cabbage, radish, garlic paste, ginger paste, salt, and gochujang.',
 '2.Cook': 'Inferior jars filled with salted cabbage mixture are turned occasionally and fermented for several days at room temperature.',
 '3.Finish': 'Served chilled, often garnished with chopped green onions and sometimes served with side dishes like kimchi pancakes or spicy beef stew.'}

In [69]:
#최종 체인 : 나라이름 -> 그 나라에 가장 유명한 음식의 레시피
final_chain = famous_food_chain | recipe_chain
final_chain.invoke({'국가':'일본'})

{'food': '사시미 (Sashimi)',
 '1.Prep': 'Select fresh seafood (usually fish) and thinly slice against the grain.',
 '2.Cook': 'No cooking involved; served raw for optimal freshness and flavor.',
 '3.Finish': 'Served on a platter with soy sauce, wasabi, and pickled ginger.'}